# SpaceX Falcon 9 API Data Collection


This notebook analyzes SpaceX Falcon 9 launch data to explore factors affecting first-stage landing success and build predictive models.


## Objective
- Build a cleaned Falcon 9 launch dataset from the SpaceX API.
- Enrich launch records with booster, payload, core, and launch site details.
- Save the modeling-ready result to `data/dataset_part_1.csv`.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (candidate / "data").exists() and (candidate / "notebooks").exists()
)
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOKS_DIR))

import datetime
import json

import pandas as pd
import requests

from notebook_utils import data_path, ensure_text_file


## Data Loading


In [2]:
API_BASE_URL = "https://api.spacexdata.com/v4"
API_SNAPSHOT_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json"
)
CACHE_DIR = data_path("api_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = data_path("dataset_part_1.csv")
rebuild_from_api = False


def load_cached_json(cache_name: str, url: str):
    cache_path = CACHE_DIR / cache_name
    if not cache_path.exists():
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        cache_path.write_text(response.text, encoding="utf-8")
    return json.loads(cache_path.read_text(encoding="utf-8"))


snapshot_path = ensure_text_file("API_call_spacex_api.json", API_SNAPSHOT_URL)
launch_records = pd.read_json(snapshot_path)
launch_records.head()


,fairings,links,static_fire_date_utc,static_fire_date_unix,tbd,net,window,rocket,success,details,...,failures,flight_number,name,date_utc,date_unix,date_local,date_precision,upcoming,cores,id
0,"{'reused': False, 'recovery_attempt': False, '...",{'patch': {'small': 'https://images2.imgbox.co...,2006-03-17T00:00:00.000Z,1.142554e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Engine failure at 33 seconds and loss of vehicle,...,"[{'time': 33, 'altitude': None, 'reason': 'mer...",1,FalconSat,2006-03-24T22:30:00.000Z,1143239400,2006-03-25T10:30:00+12:00,hour,False,"[{'core': '5e9e289df35918033d3b2623', 'flight'...",5eb87cd9ffd86e000604b32a
1,"{'reused': False, 'recovery_attempt': False, '...",{'patch': {'small': 'https://images2.imgbox.co...,None,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Successful first stage burn and transition to ...,...,"[{'time': 301, 'altitude': 289, 'reason': 'har...",2,DemoSat,2007-03-21T01:10:00.000Z,1174439400,2007-03-21T13:10:00+12:00,hour,False,"[{'core': '5e9e289ef35918416a3b2624', 'flight'...",5eb87cdaffd86e000604b32b
2,"{'reused': False, 'recovery_attempt': False, '...",{'patch': {'small': 'https://images2.imgbox.co...,None,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Residual stage 1 thrust led to collision betwe...,...,"[{'time': 140, 'altitude': 35, 'reason': 'resi...",3,Trailblazer,2008-08-03T03:34:00.000Z,1217734440,2008-08-03T15:34:00+12:00,hour,False,"[{'core': '5e9e289ef3591814873b2625', 'flight'...",5eb87cdbffd86e000604b32c
3,"{'reused': False, 'recovery_attempt': False, '...",{'patch': {'small': 'https://images2.imgbox.co...,2008-09-20T00:00:00.000Z,1.221869e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,True,Ratsat was carried to orbit on the first succe...,...,[],4,RatSat,2008-09-28T23:15:00.000Z,1222643700,2008-09-28T11:15:00+12:00,hour,False,"[{'core': '5e9e289ef3591855dc3b2626', 'flight'...",5eb87cdbffd86e000604b32d
4,"{'reused': False, 'recovery_attempt': False, '...",{'patch': {'small': 'https://images2.imgbox.co...,None,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,True,None,...,[],5,RazakSat,2009-07-13T03:35:00.000Z,1247456100,2009-07-13T15:35:00+12:00,hour,False,"[{'core': '5e9e289ef359184f103b2627', 'flight'...",5eb87cdcffd86e000604b32e


## Data Cleaning


In [3]:
launch_records = launch_records[["rocket", "payloads", "launchpad", "cores", "flight_number", "date_utc"]]
launch_records = launch_records[launch_records["cores"].map(len) == 1]
launch_records = launch_records[launch_records["payloads"].map(len) == 1].copy()
launch_records["cores"] = launch_records["cores"].map(lambda value: value[0])
launch_records["payloads"] = launch_records["payloads"].map(lambda value: value[0])
launch_records["date"] = pd.to_datetime(launch_records["date_utc"]).dt.date
launch_records = launch_records[launch_records["date"] <= datetime.date(2020, 11, 13)].copy()
launch_records.head()


,rocket,payloads,launchpad,cores,flight_number,date_utc,date
0,5e9d0d95eda69955f709d1eb,5eb0e4b5b6c3bb0006eeb1e1,5e9e4502f5090995de566f86,"{'core': '5e9e289df35918033d3b2623', 'flight':...",1,2006-03-24T22:30:00.000Z,2006-03-24
1,5e9d0d95eda69955f709d1eb,5eb0e4b6b6c3bb0006eeb1e2,5e9e4502f5090995de566f86,"{'core': '5e9e289ef35918416a3b2624', 'flight':...",2,2007-03-21T01:10:00.000Z,2007-03-21
3,5e9d0d95eda69955f709d1eb,5eb0e4b7b6c3bb0006eeb1e5,5e9e4502f5090995de566f86,"{'core': '5e9e289ef3591855dc3b2626', 'flight':...",4,2008-09-28T23:15:00.000Z,2008-09-28
4,5e9d0d95eda69955f709d1eb,5eb0e4b7b6c3bb0006eeb1e6,5e9e4502f5090995de566f86,"{'core': '5e9e289ef359184f103b2627', 'flight':...",5,2009-07-13T03:35:00.000Z,2009-07-13
5,5e9d0d95eda69973a809d1ec,5eb0e4b7b6c3bb0006eeb1e7,5e9e4501f509094ba4566f84,"{'core': '5e9e289ef359185f2b3b2628', 'flight':...",6,2010-06-04T18:45:00.000Z,2010-06-04


## Feature Engineering


In [4]:
def load_resource(resource: str, resource_id: str):
    cache_name = f"{resource}_{resource_id}.json"
    return load_cached_json(cache_name, f"{API_BASE_URL}/{resource}/{resource_id}")


def build_dataset_from_api(df: pd.DataFrame) -> pd.DataFrame:
    booster_version = [load_resource("rockets", rocket_id)["name"] for rocket_id in df["rocket"]]

    launch_sites, longitudes, latitudes = [], [], []
    payload_mass, orbit = [], []
    outcomes, flights, gridfins, reused, legs = [], [], [], [], []
    landing_pad, blocks, reused_count, serials = [], [], [], []

    for launchpad_id in df["launchpad"]:
        launchpad_data = load_resource("launchpads", launchpad_id)
        launch_sites.append(launchpad_data["name"])
        longitudes.append(launchpad_data["longitude"])
        latitudes.append(launchpad_data["latitude"])

    for payload_id in df["payloads"]:
        payload_data = load_resource("payloads", payload_id)
        payload_mass.append(payload_data["mass_kg"])
        orbit.append(payload_data["orbit"])

    for core in df["cores"]:
        core_data = load_resource("cores", core["core"])
        outcomes.append(f"{core_data['landing_success']} {core_data['landing_type']}")
        flights.append(core_data["flight"])
        gridfins.append(core["gridfins"])
        reused.append(core["reused"])
        legs.append(core["legs"])
        landing_pad.append(core["landpad"])
        blocks.append(core_data["block"])
        reused_count.append(core_data["reuse_count"])
        serials.append(core_data["serial"])

    launch_df = pd.DataFrame(
        {
            "FlightNumber": df["flight_number"].to_list(),
            "Date": df["date"].to_list(),
            "BoosterVersion": booster_version,
            "PayloadMass": payload_mass,
            "Orbit": orbit,
            "LaunchSite": launch_sites,
            "Outcome": outcomes,
            "Flights": flights,
            "GridFins": gridfins,
            "Reused": reused,
            "Legs": legs,
            "LandingPad": landing_pad,
            "Block": blocks,
            "ReusedCount": reused_count,
            "Serial": serials,
            "Longitude": longitudes,
            "Latitude": latitudes,
        }
    )

    falcon9_df = launch_df[launch_df["BoosterVersion"] != "Falcon 1"].copy()
    falcon9_df.loc[:, "FlightNumber"] = range(1, len(falcon9_df) + 1)
    falcon9_df.loc[:, "PayloadMass"] = falcon9_df["PayloadMass"].fillna(falcon9_df["PayloadMass"].mean())
    return falcon9_df


In [5]:
if rebuild_from_api or not OUTPUT_PATH.exists():
    falcon9_df = build_dataset_from_api(launch_records)
    falcon9_df.to_csv(OUTPUT_PATH, index=False)
else:
    falcon9_df = pd.read_csv(OUTPUT_PATH)

falcon9_df.head()


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


## Key Findings


In [6]:
falcon9_df[["LaunchSite", "Outcome"]].head()


,LaunchSite,Outcome
0,CCAFS SLC 40,None None
1,CCAFS SLC 40,None None
2,CCAFS SLC 40,None None
3,VAFB SLC 4E,False Ocean
4,CCAFS SLC 40,None None


In [7]:
OUTPUT_PATH


PosixPath('/Users/riyana/Desktop/spacex/data/dataset_part_1.csv')